In [5]:
# %pip install --upgrade transformers
%pip install -U accelerate transformers[torch]
# %pip install -U transformers
# %pip uninstall -y transformers

# %pip install datasets transformers[sentencepiece]
# %pip install sentencepiece
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd, json

In [3]:
pii_dataset = pd.read_csv('processed_pii_dataset.csv', usecols=['text', 'json'], nrows=1000)
pii_dataset.head()

,text,json
0,"My name is Aaliyah Popova, and I am a jeweler ...","{""name"": ""Aaliyah Popova"", ""email"": ""aaliyah.p..."
1,"My name is Konstantin Becker, and I'm a develo...","{""name"": ""Konstantin Becker"", ""email"": ""konsta..."
2,"As Mieko Mitsubishi, an account manager at a p...","{""name"": ""Mieko Mitsubishi"", ""email"": ""mieko_m..."
3,"My name is Kazuo Sun, and I'm an air traffic c...","{""name"": ""Kazuo Sun"", ""email"": ""kazuosun@hotma..."
4,"My name is Arina Sun, and I'm a dental hygieni...","{""name"": ""Arina Sun"", ""email"": ""arina-sun@gmai..."


In [4]:
pii_dataset['target_json'] = pii_dataset['json']
pii_dataset = pii_dataset.rename(columns={"text": "input_text"})  # Rename for clarity
pii_dataset

,input_text,json,target_json
0,"My name is Aaliyah Popova, and I am a jeweler ...","{""name"": ""Aaliyah Popova"", ""email"": ""aaliyah.p...","{""name"": ""Aaliyah Popova"", ""email"": ""aaliyah.p..."
1,"My name is Konstantin Becker, and I'm a develo...","{""name"": ""Konstantin Becker"", ""email"": ""konsta...","{""name"": ""Konstantin Becker"", ""email"": ""konsta..."
2,"As Mieko Mitsubishi, an account manager at a p...","{""name"": ""Mieko Mitsubishi"", ""email"": ""mieko_m...","{""name"": ""Mieko Mitsubishi"", ""email"": ""mieko_m..."
3,"My name is Kazuo Sun, and I'm an air traffic c...","{""name"": ""Kazuo Sun"", ""email"": ""kazuosun@hotma...","{""name"": ""Kazuo Sun"", ""email"": ""kazuosun@hotma..."
4,"My name is Arina Sun, and I'm a dental hygieni...","{""name"": ""Arina Sun"", ""email"": ""arina-sun@gmai...","{""name"": ""Arina Sun"", ""email"": ""arina-sun@gmai..."
...,...,...,...
995,"Lucy Lange, a dedicated and passionate nutriti...","{""name"": ""Lucy Lange"", ""email"": ""lucy_lange@ao...","{""name"": ""Lucy Lange"", ""email"": ""lucy_lange@ao..."
996,"Krishna Weber, an architect with a keen eye fo...","{""name"": ""Krishna Weber"", ""email"": ""krishna-we...","{""name"": ""Krishna Weber"", ""email"": ""krishna-we..."
997,"As Kash Weiss, a seasoned butcher with years o...","{""name"": ""Kash Weiss"", ""email"": ""kash_weiss521...","{""name"": ""Kash Weiss"", ""email"": ""kash_weiss521..."
998,In the realm of intellectual curiosity and aca...,"{""name"": ""Justin Schmitt"", ""email"": ""justinsch...","{""name"": ""Justin Schmitt"", ""email"": ""justinsch..."


In [5]:
from datasets import Dataset
pii_dataset = Dataset.from_pandas(pii_dataset[['input_text', 'target_json']])
pii_dataset = pii_dataset.train_test_split(test_size=0.2)


/home/bilal/Learning_Paths/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [39]:
pii_dataset['train']["input_text"]

Column(["My name is Konstantin Becker, and I'm a developer with two years of experience. I recently worked on a project where we built a new customer portal for our company. The goal was to create a more user-friendly and intuitive interface that would make it easier for customers to manage their accounts and access information. We started by gathering requirements from our customers and conducting user research to understand their needs and pain points. We then designed a new user interface that was both visually appealing and easy to use. We also implemented a number of new features, such as the ability for customers to view their account history, track their orders, and submit support tickets online. The project was a success, and our customers were very happy with the new portal. They found it to be much easier to use than the old one, and they appreciated the new features. We saw a significant increase in customer satisfaction and engagement as a result of the new portal. Througho

In [6]:
from transformers import T5Tokenizer

# tokenizer = T5Tokenizer.from_pretrained("t5-large")
tokenizer = T5Tokenizer.from_pretrained('./model/t5_tokenizer')

max_input_length = 512
max_target_length = 256

def preprocess(example):
    input_text = example["input_text"]
    target_text = example["target_json"]
    
    model_input = tokenizer(
        input_text,
        max_length=max_input_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            target_text,
            max_length=max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    model_input["labels"] = labels["input_ids"][0]
    model_input["attention_mask"] = model_input["attention_mask"][0]
    model_input["input_ids"] = model_input["input_ids"][0]
    # print(model_input)
    return model_input
tokenized_dataset = {}
tokenized_dataset = pii_dataset.map(preprocess)



Map:   0%|          | 0/800 [00:00<?, ? examples/s]/home/bilal/Learning_Paths/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 200/200 [00:00<00:00, 311.32 examples/s]


In [61]:
print("Input ID", tokenized_dataset['train']['input_ids'])
print("Labels: ", tokenized_dataset['train']['labels'])

Input ID Column([[499, 564, 19, 2974, 7, 3672, 77, 9864, 49, 6, 11, 27, 31, 51, 3, 9, 7523, 28, 192, 203, 13, 351, 5, 27, 1310, 1279, 30, 3, 9, 516, 213, 62, 1192, 3, 9, 126, 884, 8948, 21, 69, 349, 5, 37, 1288, 47, 12, 482, 3, 9, 72, 1139, 18, 4905, 11, 12954, 3459, 24, 133, 143, 34, 1842, 21, 722, 12, 1865, 70, 3744, 11, 592, 251, 5, 101, 708, 57, 7241, 1502, 45, 69, 722, 11, 13646, 1139, 585, 12, 734, 70, 523, 11, 1406, 979, 5, 101, 258, 876, 3, 9, 126, 1139, 3459, 24, 47, 321, 17307, 12817, 11, 514, 12, 169, 5, 101, 92, 6960, 3, 9, 381, 13, 126, 753, 6, 224, 38, 8, 1418, 21, 722, 12, 903, 70, 905, 892, 6, 1463, 70, 5022, 6, 11, 4237, 380, 3500, 367, 5, 37, 516, 47, 3, 9, 1269, 6, 11, 69, 722, 130, 182, 1095, 28, 8, 126, 8948, 5, 328, 435, 34, 12, 36, 231, 1842, 12, 169, 145, 8, 625, 80, 6, 11, 79, 9332, 8, 126, 753, 5, 101, 1509, 3, 9, 1516, 993, 16, 884, 5044, 11, 3813, 38, 3, 9, 741, 13, 8, 126, 8948, 5, 3, 11714, 8, 516, 6, 27, 47, 1966, 21, 2421, 8, 223, 18, 989, 1081, 21, 8, 8

In [7]:
from transformers import T5ForConditionalGeneration

# model = T5ForConditionalGeneration.from_pretrained("t5-large")
model = T5ForConditionalGeneration.from_pretrained('./model/t5_model')
# tokenizer = T5ForConditionalGeneration.from_pretrained('./model/t5_model')


# tokenizer.save_pretrained('./model/t5_tokenizer')
# model.save_pretrained('./model/t5_model')

In [62]:
tokenized_dataset['train'][0]

{'input_text': "My name is Konstantin Becker, and I'm a developer with two years of experience. I recently worked on a project where we built a new customer portal for our company. The goal was to create a more user-friendly and intuitive interface that would make it easier for customers to manage their accounts and access information. We started by gathering requirements from our customers and conducting user research to understand their needs and pain points. We then designed a new user interface that was both visually appealing and easy to use. We also implemented a number of new features, such as the ability for customers to view their account history, track their orders, and submit support tickets online. The project was a success, and our customers were very happy with the new portal. They found it to be much easier to use than the old one, and they appreciated the new features. We saw a significant increase in customer satisfaction and engagement as a result of the new portal. T

In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./t5-json-model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    do_predict=True,
    fp16=True,  # if you're using a GPU with mixed precision support
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
)

trainer.train()


/tmp/ipykernel_1913/3544683791.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 10.95 GiB is allocated by PyTorch, and 12.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
import transformers
transformers.__version__

'4.57.1'